# TALENTIQ — WHAT PREPROCESSING AND FEATURE ENGINEERING ACTUALLY DID


### this notebook does not clean anything by itself
the real cleaning already happened inside src/preprocessing.py and src/feature_engineering.py when those files were run. here we are just loading what they produced and walking through it, so anyone reading this can follow what happened without going line by line through the source code.


## PART 1 — PREPROCESSING (src/preprocessing.py)


### -- what was wrong with the raw data
straight out of the CSV, the resume dataset had four problems we needed to deal with before any model could use it.

1. duplicate rows — same candidate showing up more than once, which would let the model just memorise that row instead of learning a general pattern
2. missing values — empty cells, and a model cannot work with an empty cell, it needs a number every time
3. outliers — a few candidates with unrealistic values, like years of experience that don't make sense, which can pull the model's decision boundary off in a bad direction
4. text columns — education_level, company_type and the rest are words, not numbers, and models only work with numbers


### -- step 1, removed duplicate rows
any row that was an exact copy of another one was dropped. straightforward, no real decision to make here.

### -- step 2, filled in the missing values
for number columns like cgpa and skills_score, the gaps were filled with the median of that column. median instead of mean because it does not get pulled around by outliers.
for category columns like education_level, the gaps were filled with the mode, basically the most common value, which is the safest guess when there is nothing else to go on.

### -- step 3, capped the outliers using the IQR rule
for cgpa, experience_years, skills_score and soft_skills_score, we worked out a normal range using IQR. anything way outside the middle 50% of the data is treated as suspicious. instead of deleting these rows, the values were clipped down to the nearest acceptable boundary, so we keep the candidate but stop that one value from dragging the model around.

### -- step 4, encoding and scaling happen inside a ColumnTransformer in train.py
rather than manually encoding and then scaling as separate steps, both transformations are wrapped into a single ColumnTransformer object inside train.py. this ensures the same sequence is always applied in the same order, on both training and test data.

education_level and university_tier have a natural order (High School < Bachelor < Master < PhD, Tier 3 < Tier 2 < Tier 1), so OrdinalEncoder is used with the explicit category order from features.yaml.
company_type has no natural order, so OneHotEncoder is used, which gives each category its own 0/1 column.
StandardScaler is applied to all numerical columns, bringing them to mean=0 and std=1 so no feature dominates just because its raw numbers are larger.

### -- one important detail, the split happened before any fitting
the ColumnTransformer was fit only on the training data, then the exact same fitted transformer was applied to the test data. if we had fit on the full dataset first, the test set would leak information into training, and our evaluation numbers later would look better than they would be in the real world.

### -- what got saved to disk and why
- artifacts/preprocessor.pkl — the entire fitted ColumnTransformer (encoders + scaler together), so the exact same transformation can be applied to a new candidate at inference time
- data/splits/X_train.csv, X_test.csv, y_train.csv, y_test.csv — the cleaned splits ready for modelling


### -- let's actually look at the result


In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import joblib
from src.config_loader import load_config, load_features

cfg = load_config()
feat_cfg = load_features()
target = feat_cfg["target"]  # 'hired'

X_train = pd.read_csv("../data/splits/X_train.csv")
X_test  = pd.read_csv("../data/splits/X_test.csv")
y_train = pd.read_csv("../data/splits/y_train.csv")
y_test  = pd.read_csv("../data/splits/y_test.csv")

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")
print()
print("First few rows of training features:")
X_train.head()


In [ ]:
# load the fitted preprocessor and inspect what's inside it
preprocessor = joblib.load("../artifacts/preprocessor.pkl")

print("Preprocessor transformers:")
for name, transformer, columns in preprocessor.transformers:
    print(f"  {name}: {transformer.__class__.__name__} -> {columns}")


### -- this confirms the preprocessor was saved correctly
we are loading preprocessor.pkl and printing each transformer inside it. ordinal handles the ordered categories, onehot handles company_type, scaler handles all numerical columns. this is the exact same object that will be loaded at inference time to transform a new candidate.


## PART 2 — FEATURE ENGINEERING (src/feature_engineering.py)


### -- why we built new features instead of just using the raw columns
the raw columns describe a candidate, but they don't directly capture some bigger picture ideas that probably matter for hiring, like how employable someone is overall, or how complete their profile is. rather than hoping the model figures these patterns out by itself from the raw numbers, 7 new columns were hand built that bring in this domain knowledge directly.


### -- the 7 features, and the thinking behind each one

**EmployabilityScore** = 0.3 x cgpa + 0.4 x skills_score + 0.3 x soft_skills_score
one combined score, weighted more toward skills (0.4) than academics or soft skills (0.3 each), since skills usually matter more than GPA when it comes to hiring.

**PortfolioStrength** = projects x 0.5 + hackathons x 0.3 + research_papers x 0.2
measures hands-on proof of ability. projects get the highest weight since they are the most direct evidence of applied skill.

**TechnicalReadiness** = programming_languages x 0.4 + certifications x 0.3 + skills_score x 0.3
mixes breadth (how many languages someone knows) with depth (certifications and skill score) into one number.

**ExperienceQuality** = (experience_years x 0.6 + internships x 0.4), then min-max normalized to 0-1
years of experience matters most here, but internships still count. normalizing to 0-1 makes it easier to compare against the other scores. a zero-division guard is in place in case all candidates have identical raw values.

**LearningIndex** = certifications + research_papers + hackathons
a simple sum. all three point toward a habit of continuous learning, so there was no need to weight them differently.

**ProfileCompleteness** = count of non-null fields per candidate row
a thin or incomplete profile might be its own signal, either inexperience or just low effort on the application.

**SkillExperienceGap** = abs(skills_score - experience_years x 10)
flags a mismatch, like a high skill score with very little experience, or the other way round, which can be a useful signal on its own.

the last two (ProfileCompleteness and SkillExperienceGap) were added on top of the original five, specifically to catch patterns the first five did not cover.

### -- why hand build these instead of letting the model find the patterns itself
simpler models like Logistic Regression can only combine the raw features in fairly basic ways. by pre-combining related columns into one meaningful score ahead of time, these patterns become much easier for every model to actually use, not just the more complex ones.


### -- let's actually look at the result


In [ ]:
mode = cfg["mode"]
processed_path = "../" + cfg["paths"]["processed"][mode]
cleaned_df = pd.read_csv(processed_path)

feature_cols = joblib.load("../artifacts/feature_columns.pkl")

engineered = [
    "EmployabilityScore", "PortfolioStrength", "TechnicalReadiness",
    "ExperienceQuality", "LearningIndex", "ProfileCompleteness", "SkillExperienceGap"
]

print(f"Total feature columns saved for modeling: {len(feature_cols)}")
print()
print("The 7 engineered features, with summary stats:")
cleaned_df[engineered].describe()


In [ ]:
# quick sanity check — do these engineered features actually relate to hired?
if target in cleaned_df.columns:
    print("Average value of each engineered feature, split by hired outcome:")
    print(cleaned_df.groupby(target)[engineered].mean())
else:
    print(f"Target column '{target}' not found in processed file. Run the pipeline first.")


### -- a quick sanity check on the new features
grouping by hired and averaging each engineered feature lets us see straight away whether these scores actually move in a sensible direction between the two outcomes, a useful gut-check before trusting them in modelling.


## SUMMARY — WHAT TO TAKE AWAY

- preprocessing fixed the data quality problems — duplicates removed, missing values filled sensibly, outliers capped instead of deleted
- encoding and scaling are handled inside a single ColumnTransformer (preprocessor.pkl), so the same transformation is guaranteed to run in the same order at both training time and inference time
- feature engineering added domain knowledge on top — 7 new columns that pre-combine related raw signals into single, more meaningful scores, so the model does not have to discover these combinations on its own
- none of this was decided by EDA alone — EDA only pointed out that problems existed, the actual fix and the formulas were design decisions made inside these two files
- what this notebook does NOT do — it does not re-run preprocessing or feature engineering, it only loads what those scripts already produced and walks through it. if the source files change, run the pipeline again first, then re-run this notebook to see the updated result
